In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
import logging
import sys
import pickle

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, accuracy_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE

In [29]:
# logging configuration
LOG_DIR = "ml_logs"
os.makedirs(LOG_DIR, exist_ok=True)
LOG_FILE = os.path.join(LOG_DIR, "heart_disease_model.log")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler()
    ]
)

logging.info("Logging configured. Log File: {LOG_FILE}")


2025-10-26 16:16:06,664 - INFO - Logging configured. Log File: {LOG_FILE}


In [30]:
df = pd.read_csv(r"C:\Users\Priya Bhaskar\practice_ml_projects\heart_disease_prediction\hdp\data\framingham.csv")
logging.info("Dataset loaded successfully.")

2025-10-26 16:16:07,235 - INFO - Dataset loaded successfully.


In [31]:
df

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4233,1,50,1.0,1,1.0,0.0,0,1,0,313.0,179.0,92.0,25.97,66.0,86.0,1
4234,1,51,3.0,1,43.0,0.0,0,0,0,207.0,126.5,80.0,19.71,65.0,68.0,0
4235,0,48,2.0,1,20.0,NaN,0,0,0,248.0,131.0,72.0,22.00,84.0,86.0,0
4236,0,44,1.0,1,15.0,0.0,0,0,0,210.0,126.5,87.0,19.16,86.0,NaN,0


In [32]:
if df is not None:
    logging.info(f"Dataset first 5 rows: {df.head()}")
    logging.info("Column information and data types:")
    df.info()
    logging.info("Statistical summary of the numeric columns:")
    print(df.describe())

2025-10-26 16:16:08,191 - INFO - Dataset first 5 rows:    male  age  education  currentSmoker  cigsPerDay  BPMeds  prevalentStroke  \
0     1   39        4.0              0         0.0     0.0                0   
1     0   46        2.0              0         0.0     0.0                0   
2     1   48        1.0              1        20.0     0.0                0   
3     0   61        3.0              1        30.0     0.0                0   
4     0   46        3.0              1        23.0     0.0                0   

   prevalentHyp  diabetes  totChol  sysBP  diaBP    BMI  heartRate  glucose  \
0             0         0    195.0  106.0   70.0  26.97       80.0     77.0   
1             0         0    250.0  121.0   81.0  28.73       95.0     76.0   
2             0         0    245.0  127.5   80.0  25.34       75.0     70.0   
3             1         0    225.0  150.0   95.0  28.58       65.0    103.0   
4             0         0    285.0  130.0   84.0  23.10       85.0     85.0

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4238 entries, 0 to 4237
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   male             4238 non-null   int64  
 1   age              4238 non-null   int64  
 2   education        4133 non-null   float64
 3   currentSmoker    4238 non-null   int64  
 4   cigsPerDay       4209 non-null   float64
 5   BPMeds           4185 non-null   float64
 6   prevalentStroke  4238 non-null   int64  
 7   prevalentHyp     4238 non-null   int64  
 8   diabetes         4238 non-null   int64  
 9   totChol          4188 non-null   float64
 10  sysBP            4238 non-null   float64
 11  diaBP            4238 non-null   float64
 12  BMI              4219 non-null   float64
 13  heartRate        4237 non-null   float64
 14  glucose          3850 non-null   float64
 15  TenYearCHD       4238 non-null   int64  
dtypes: float64(9), int64(7)
memory usage: 529.9 KB
             

In [33]:
df.isnull().sum()

male                 0
age                  0
education          105
currentSmoker        0
cigsPerDay          29
BPMeds              53
prevalentStroke      0
prevalentHyp         0
diabetes             0
totChol             50
sysBP                0
diaBP                0
BMI                 19
heartRate            1
glucose            388
TenYearCHD           0
dtype: int64

In [34]:
if df is not None:
    logging.info("Checking for missing Values")
    missing_values = df.isnull().sum()
    missing_percentage = missing_values/len(df) * 100
    

2025-10-26 16:16:08,683 - INFO - Checking for missing Values


In [35]:
missing_percentage

male               0.000000
age                0.000000
education          2.477584
currentSmoker      0.000000
cigsPerDay         0.684285
BPMeds             1.250590
prevalentStroke    0.000000
prevalentHyp       0.000000
diabetes           0.000000
totChol            1.179802
sysBP              0.000000
diaBP              0.000000
BMI                0.448325
heartRate          0.023596
glucose            9.155262
TenYearCHD         0.000000
dtype: float64

In [36]:
if df is not None:
    try:
        missing_cols = df.columns[df.isnull().any()].tolist()
        logging.info(f"Handling missing values in columns: {missing_cols}")
        for col in missing_cols:
            if df[col].dtype in ['float64', 'int64']:
                median_val = df[col].median()
                # df[col].fillna(median_val, inplace=True)
                df[col] = df[col].fillna(median_val)

                logging.info(f"Filled missing values in '{col}' with median: {median_val}")

            else:
                mode_val = df[col].mode()[0]
                # df[col].fillna(mode_val, inplace=True)
                df[col] = df[col].fillna(mode_val)
                logging.info(f"Filled missing values in '{col}' with mode: {mode_val}")

        if df.isnull().sum() == 0:
            logging.info("Missing value imputation completed. Dataset now has no missing values")

    except Exception as e:
        logging.error(f"Error occured during missing value imputation: {e}")

    


2025-10-26 16:16:09,216 - INFO - Handling missing values in columns: ['education', 'cigsPerDay', 'BPMeds', 'totChol', 'BMI', 'heartRate', 'glucose']
2025-10-26 16:16:09,221 - INFO - Filled missing values in 'education' with median: 2.0
2025-10-26 16:16:09,222 - INFO - Filled missing values in 'cigsPerDay' with median: 0.0
2025-10-26 16:16:09,227 - INFO - Filled missing values in 'BPMeds' with median: 0.0
2025-10-26 16:16:09,228 - INFO - Filled missing values in 'totChol' with median: 234.0
2025-10-26 16:16:09,232 - INFO - Filled missing values in 'BMI' with median: 25.4
2025-10-26 16:16:09,232 - INFO - Filled missing values in 'heartRate' with median: 75.0
2025-10-26 16:16:09,237 - INFO - Filled missing values in 'glucose' with median: 78.0
2025-10-26 16:16:09,242 - ERROR - Error occured during missing value imputation: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().


In [37]:
df.isnull().sum()

male               0
age                0
education          0
currentSmoker      0
cigsPerDay         0
BPMeds             0
prevalentStroke    0
prevalentHyp       0
diabetes           0
totChol            0
sysBP              0
diaBP              0
BMI                0
heartRate          0
glucose            0
TenYearCHD         0
dtype: int64

In [38]:
# outliers
if df is not None:
    numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
    # print(numerical_cols)
    binary_cols = ['male', 'currentSmoker', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes', 'TenYearCHD']
    cols_for_outlier_handling = [col for col in numerical_cols if col not in binary_cols]

    logging.info(f"Applying outlier capping (IQR method) for columns: {cols_for_outlier_handling}")

    try:
        for col in cols_for_outlier_handling:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 -1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR

            initial_outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)].shape[0]
            if initial_outliers > 0:
                df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])
                df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])
                logging.info(f"Capped {initial_outliers} outliers in column '{col}' at [{lower_bound:.2f}, {upper_bound: .2f}] ")

            else:
                logging.info(f"No significant outliers found in the column '{col}' to cap")

    except Exception as e:
        logging.error(f"Error during the outlier handling: {e}")




2025-10-26 16:16:09,724 - INFO - Applying outlier capping (IQR method) for columns: ['age', 'education', 'cigsPerDay', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']
2025-10-26 16:16:09,728 - INFO - No significant outliers found in the column 'age' to cap
2025-10-26 16:16:09,732 - INFO - No significant outliers found in the column 'education' to cap
2025-10-26 16:16:09,737 - INFO - Capped 12 outliers in column 'cigsPerDay' at [-30.00,  50.00] 
2025-10-26 16:16:09,742 - INFO - Capped 57 outliers in column 'totChol' at [122.00,  346.00] 
2025-10-26 16:16:09,745 - INFO - Capped 126 outliers in column 'sysBP' at [76.50,  184.50] 
2025-10-26 16:16:09,752 - INFO - Capped 81 outliers in column 'diaBP' at [52.69,  112.19] 
2025-10-26 16:16:09,759 - INFO - Capped 97 outliers in column 'BMI' at [15.64,  35.47] 
2025-10-26 16:16:09,766 - INFO - Capped 76 outliers in column 'heartRate' at [45.50,  105.50] 
2025-10-26 16:16:09,770 - INFO - Capped 262 outliers in column 'glucose' at [52

In [39]:
# Visual Representation of EDA
# Target Variable Distribution

if df is not None:
    try:
        fig = px.pie(data_frame=df, names='TenYearCHD', title="Distribution of TenYearCHD", hole=0.3, color_discrete_sequence=px.colors.qualitative.Pastel)
        fig.update_traces(textinfo='percent+label', marker=dict(line=dict(color="#000000", width=1)))
        fig.show()

        logging.info("Displayed pie chart for TenYearCHD distribution")
    except Exception as e:
        logging.error(f"Error plotting TenYearCHD distribution: {e}")

2025-10-26 16:16:10,034 - INFO - Displayed pie chart for TenYearCHD distribution


In [40]:
# The above pie chart indicates the imbalanced classes of 0 (No) and 1 (yes)
# we will have to deal with this type of dataset differently.

In [41]:
# Numerical Feature Distributions
if df is not None:
    
    numerical_cols = ['age', 'cigsPerDay', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate','glucose']
    fig = make_subplots(rows=len(numerical_cols)//2 + len(numerical_cols) %2, cols=2, subplot_titles=[f"Distribution of {col}" for col in numerical_cols])

    for i, col in enumerate(numerical_cols):
        row = (i // 2) + 1
        col_idx = (i % 2) + 1
        fig.add_trace(go.Histogram(x=df[col], name=col, marker_color=px.colors.qualitative.Plotly[i%10]),row=row, col=col_idx)
        fig.update_xaxes(title_text=col, row=row, col=col_idx)
        fig.update_yaxes(title_text="Count", row=row, col=col_idx)

    fig.update_layout(height=400 * (len(numerical_cols)//2 + len(numerical_cols)%2), showlegend=False)
    fig.show()
    logging.info("Displayed histogram for numerical features")
        




2025-10-26 16:16:10,912 - INFO - Displayed histogram for numerical features


In [42]:
if df is not None:
    fig = make_subplots(rows=len(numerical_cols)//2 + len(numerical_cols)%2, cols=2, subplot_titles=[f'{col} vs TenYearCHD' for col in numerical_cols])

    for i, col in enumerate(numerical_cols):
        row = (i //2) + 1
        col_idx = (i %2) + 1
        fig.add_trace(go.Box(x=df['TenYearCHD'], y=df[col], name=col, marker_color=px.colors.qualitative.Plotly[i%10]), row=row, col=col_idx)
        fig.update_xaxes(title_text="TenYearCDH", row=row, col=col_idx)
        fig.update_yaxes(title_text=col, row=row, col=col_idx)

    fig.update_layout(height=400 * len(numerical_cols)//2 + len(numerical_cols)%2, showlegend=False,)
    fig.show()
    logging.info("Displayed the box plots to compare the numerical columns with target variable.")

2025-10-26 16:16:11,247 - INFO - Displayed the box plots to compare the numerical columns with target variable.


In [43]:
# Categorical Feature Representation
if df is not None:
    binary_cols = ['male', 'education', 'currentSmoker', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes']

    fig = make_subplots(rows=len(binary_cols)//2 + len(binary_cols)%2, cols=2, subplot_titles=[f'Distibution of {col}' for col in binary_cols])

    for i, col in enumerate(binary_cols):
        row = (i //2) + 1
        col_idx = (i % 2) + 1
        counts = df[col].value_counts().reset_index()
        counts.columns = [col, 'Count']
        fig.add_trace(go.Bar(x=counts[col].astype(str), y=counts['Count'], name=col, marker_color=px.colors.qualitative.Set3[i%10]), row=row, col=col_idx)

        fig.update_xaxes(title_text=col, row=row, col=col_idx)
        fig.update_yaxes(title_text="Count", row=row, col=col_idx)

    fig.update_layout(height=400*len(binary_cols)//2 + len(binary_cols)%2, showlegend=False)
    fig.show()

    logging.info("Displayed bar plots of binary columns")



2025-10-26 16:16:11,599 - INFO - Displayed bar plots of binary columns


In [44]:
# calculate the correlation matrix

if df is not None:
    try:
        corr_matrix = df.corr()
        fig = px.imshow(corr_matrix, text_auto=True, aspect='auto', color_continuous_scale='RdBu_r', title="Correlation matrix of Features")
        fig.update_layout(height=800, width=800)
        fig.show()
        logging.info("Displayed correlation matrix with heatmap")

    except Exception as e:
        logging.info(f"Error plotting correlation matrix: {e}")

    

2025-10-26 16:16:11,830 - INFO - Displayed correlation matrix with heatmap


In [45]:
if df is not None:
    # select the X features and y (target) features
    features = [col for col in df.columns if col != "TenYearCHD"]
    # print(features)
    X = df[features]
    y = df['TenYearCHD']
    logging.info(f"Selected {len(features)} features for modeling: {features}")
    logging.info(f"Target variable: TenYearCHD")
else:
    logging.error("Dataframe is not loaded, we cannot proceed with feature selection")


2025-10-26 16:16:12,246 - INFO - Selected 15 features for modeling: ['male', 'age', 'education', 'currentSmoker', 'cigsPerDay', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']
2025-10-26 16:16:12,248 - INFO - Target variable: TenYearCHD


In [46]:
if df is not None:
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
        logging.info(f"Data split into training (X_train: {X_train.shape}, y_train: {y_train.shape})")

        # find out numerical features for scaling
        numerical_features = X_train.select_dtypes(include=np.number).columns.tolist()

        scaler = StandardScaler()
        X_train_scaled = X_train.copy()
        X_test_scaled = X_test.copy()

        X_train_scaled[numerical_features] = scaler.fit_transform(X_train[numerical_features])
        X_test_scaled[numerical_features] = scaler.transform(X_test[numerical_features])

        logging.info("Numerical features are scaled using StandardScaler")

        # handle imbalanced dataset using SMOTE on training data
        # check the imbalance of the data.

        if y_train.value_counts(normalize=True)[0] > 0.75 or y_train.value_counts(normalize=True)[1] > 0.75:
            logging.info("Target variable is imbalanced. Applying SMOTE to the training data (target variable)")
            smote = SMOTE(random_state=42)

            X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
            logging.info(f"SMOTE Applied. orginal training shape: {X_train_scaled.shape, y_train.shape} vs Resampled training shape: {X_train_res.shape, y_train_res.shape}")
            X_train_final = X_train_res
            y_train_final = y_train_res
        else:
            logging.info("Training target variable is not imbalanced, so SMOTE is not applied")
            X_train_final = X_train_scaled
            y_train_final = y_train

    except Exception as e:
        logging.error(f"Error occured during data spliting or scaling")
    

        



        






2025-10-26 16:16:12,638 - INFO - Data split into training (X_train: (3178, 15), y_train: (3178,))
2025-10-26 16:16:12,658 - INFO - Numerical features are scaled using StandardScaler
2025-10-26 16:16:12,662 - INFO - Target variable is imbalanced. Applying SMOTE to the training data (target variable)
c:\Users\Priya Bhaskar\practice_ml_projects\heart_disease_prediction\hdp\venv\lib\site-packages\sklearn\base.py:474: FutureWarning:

`BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.

2025-10-26 16:16:12,678 - INFO - SMOTE Applied. orginal training shape: ((3178, 15), (3178,)) vs Resampled training shape: ((5390, 15), (5390,))


In [47]:
fig = px.pie(data_frame=y_train_res, names='TenYearCHD', title="Distribution of TenYearCHD", hole=0.3, color_discrete_sequence=px.colors.qualitative.Pastel)
fig.update_traces(textinfo='percent+label', marker=dict(line=dict(color="#000000", width=1)))
fig.show()

In [48]:
# modeling 
models = {
    "Logistic Regression": LogisticRegression(random_state=42, solver='liblinear'),
    "Random Forest Classifier": RandomForestClassifier(random_state=42, n_estimators=100),
    "Gradient Boosting Classifier": GradientBoostingClassifier(random_state=42, n_estimators=100),
    "Support Vector Classifier": SVC(random_state=42, probability=True)


}

trained_models = {}
for name, model in models.items():
    try:
        logging.info(f"Training {name}...")
        model.fit(X_train_final, y_train_final)
        trained_models[name] = model
        logging.info(f"{name} trained successfully")
    except Exception as e:
        logging.error(f"Error training {name}: {e}")

2025-10-26 16:16:14,520 - INFO - Training Logistic Regression...
2025-10-26 16:16:14,538 - INFO - Logistic Regression trained successfully
2025-10-26 16:16:14,541 - INFO - Training Random Forest Classifier...
2025-10-26 16:16:16,065 - INFO - Random Forest Classifier trained successfully
2025-10-26 16:16:16,065 - INFO - Training Gradient Boosting Classifier...
2025-10-26 16:16:18,426 - INFO - Gradient Boosting Classifier trained successfully
2025-10-26 16:16:18,426 - INFO - Training Support Vector Classifier...
2025-10-26 16:16:29,375 - INFO - Support Vector Classifier trained successfully


In [49]:
results = {}
roc_curves = {}

for name, model in trained_models.items():
    try:
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else [0] * len(y_test)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        roc_auc = roc_auc_score(y_test, y_proba) if hasattr(model, 'predict_proba') else 0

        results[name] = {
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1,
            'ROC AUC': roc_auc
        }
        logging.info(f"Evaluation for {name}:")
        logging.info(f"  Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1-Score: {f1:.4f}, ROC AUC: {roc_auc:.4f}")

        # Store ROC curve data
        if hasattr(model, 'predict_proba'):
            fpr, tpr, _ = roc_curve(y_test, y_proba)
            roc_curves[name] = {'fpr': fpr, 'tpr': tpr, 'auc': roc_auc}

        # Confusion Matrix
        cm = confusion_matrix(y_test, y_pred)
        logging.info(f"  Confusion Matrix for {name}:\n{cm}")

    except Exception as e:
        logging.error(f"Error evaluating {name}: {e}")

# Display results in a DataFrame
results_df = pd.DataFrame(results).T
print("\n--- Model Evaluation Results ---")
print(results_df.sort_values(by='F1-Score', ascending=False))

# Plot ROC curves
fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Random Classifier (AUC = 0.50)',
                                line=dict(dash='dash', color='gray')))

for name, data in roc_curves.items():
    fig_roc.add_trace(go.Scatter(x=data['fpr'], y=data['tpr'], mode='lines',
                                    name=f'{name} (AUC = {data["auc"]:.2f})'))

fig_roc.update_layout(title='ROC Curve Comparison',
                        xaxis_title='False Positive Rate',
                        yaxis_title='True Positive Rate',
                        xaxis=dict(range=[0, 1]), yaxis=dict(range=[0, 1]),
                        showlegend=True)
fig_roc.show()
logging.info("Displayed ROC curve comparison for all models.")

logging.error("Test data or trained models not available for evaluation.")

2025-10-26 16:16:29,436 - INFO - Evaluation for Logistic Regression:
2025-10-26 16:16:29,436 - INFO -   Accuracy: 0.6557, Precision: 0.2371, Recall: 0.5714, F1-Score: 0.3352, ROC AUC: 0.6981
2025-10-26 16:16:29,445 - INFO -   Confusion Matrix for Logistic Regression:
[[603 296]
 [ 69  92]]
2025-10-26 16:16:29,533 - INFO - Evaluation for Random Forest Classifier:
2025-10-26 16:16:29,533 - INFO -   Accuracy: 0.8057, Precision: 0.2472, Recall: 0.1366, F1-Score: 0.1760, ROC AUC: 0.6243
2025-10-26 16:16:29,533 - INFO -   Confusion Matrix for Random Forest Classifier:
[[832  67]
 [139  22]]
2025-10-26 16:16:29,558 - INFO - Evaluation for Gradient Boosting Classifier:
2025-10-26 16:16:29,558 - INFO -   Accuracy: 0.7736, Precision: 0.2717, Recall: 0.2919, F1-Score: 0.2814, ROC AUC: 0.6450
2025-10-26 16:16:29,568 - INFO -   Confusion Matrix for Gradient Boosting Classifier:
[[773 126]
 [114  47]]
2025-10-26 16:16:30,933 - INFO - Evaluation for Support Vector Classifier:
2025-10-26 16:16:30,933 


--- Model Evaluation Results ---
                              Accuracy  Precision    Recall  F1-Score  \
Logistic Regression           0.655660   0.237113  0.571429  0.335155   
Support Vector Classifier     0.691509   0.233974  0.453416  0.308668   
Gradient Boosting Classifier  0.773585   0.271676  0.291925  0.281437   
Random Forest Classifier      0.805660   0.247191  0.136646  0.176000   

                               ROC AUC  
Logistic Regression           0.698091  
Support Vector Classifier     0.631772  
Gradient Boosting Classifier  0.645002  
Random Forest Classifier      0.624303  


2025-10-26 16:16:30,982 - INFO - Displayed ROC curve comparison for all models.
2025-10-26 16:16:30,982 - ERROR - Test data or trained models not available for evaluation.


In [50]:
if 'X_test_scaled' in locals() and 'y_test' in locals() and trained_models:
    logging.info("Visualizing Confusion Matrices for the models.")

    num_models = len(trained_models)
    rows = (num_models + 1) // 2
    cols = 2 if num_models > 0 else 1 # Ensure at least 1 column for 0 or 1 models

    fig = make_subplots(rows=rows, cols=cols, subplot_titles=[f'Confusion Matrix: {name}' for name in trained_models.keys()])

    for i, (name, model) in enumerate(trained_models.items()):
        row = (i // cols) + 1
        col = (i % cols) + 1
        try:
            y_pred = model.predict(X_test_scaled)
            cm = confusion_matrix(y_test, y_pred)
            
            # Create a heatmap for the confusion matrix
            z = [[cm[0,0], cm[0,1]], [cm[1,0], cm[1,1]]]
            x = ['Predicted 0', 'Predicted 1']
            y = ['Actual 0', 'Actual 1']
            
            heatmap_trace = go.Heatmap(z=z, x=x, y=y, colorscale='Blues',
                                       text=[[str(val) for val in row_cm] for row_cm in cm],
                                       texttemplate="%{text}", textfont={"size":15})
            fig.add_trace(heatmap_trace, row=row, col=col)

            # Update layout to center titles
            fig.update_layout(height=400 * rows, width=400 * cols)
            fig.update_xaxes(title_text="Predicted Class", row=row, col=col)
            fig.update_yaxes(title_text="Actual Class", row=row, col=col)

        except Exception as e:
            logging.error(f"Error generating confusion matrix for {name}: {e}")

    fig.update_layout(title_text='Confusion Matrices for Different Models')
    fig.show()
    logging.info("Displayed confusion matrices for all models.")

    # A more advanced "residual" analysis could involve looking at the distributions of features for FP/FN
    # Example for one model (e.g., Random Forest)
    if 'Random Forest Classifier' in trained_models:
        rf_model = trained_models['Random Forest Classifier']
        y_pred_rf = rf_model.predict(X_test_scaled)

        # Get misclassified samples
        misclassified_indices = y_test[y_test != y_pred_rf].index
        misclassified_df = df.loc[misclassified_indices]
        misclassified_actual = y_test.loc[misclassified_indices]
        misclassified_predicted = pd.Series(y_pred_rf, index=y_test.index).loc[misclassified_indices]

        logging.info(f"\n--- Analysis of Misclassified Samples (Random Forest) ---")
        logging.info(f"Total misclassified samples: {len(misclassified_indices)}")
        print("Misclassified Samples (Actual vs Predicted):\n", pd.DataFrame({'Actual': misclassified_actual, 'Predicted': misclassified_predicted}).head())

        # Further analysis: distributions of key features for FP vs FN
        # FP: Actual 0, Predicted 1
        fp_indices = y_test[(y_test == 0) & (y_pred_rf == 1)].index
        fp_df = df.loc[fp_indices]
        # FN: Actual 1, Predicted 0
        fn_indices = y_test[(y_test == 1) & (y_pred_rf == 0)].index
        fn_df = df.loc[fn_indices]

        logging.info(f"Number of False Positives (Actual 0, Predicted 1): {len(fp_df)}")
        logging.info(f"Number of False Negatives (Actual 1, Predicted 0): {len(fn_df)}")

        # Example: Compare 'age' distribution for FP and FN
        if not fp_df.empty and not fn_df.empty:
            fig_err_age = go.Figure()
            fig_err_age.add_trace(go.Violin(y=fp_df['age'], name='False Positives (Age)', box_visible=True, meanline_visible=True, fillcolor='lightblue', line_color='blue'))
            fig_err_age.add_trace(go.Violin(y=fn_df['age'], name='False Negatives (Age)', box_visible=True, meanline_visible=True, fillcolor='lightcoral', line_color='red'))
            fig_err_age.update_layout(title='Age Distribution of False Positives vs. False Negatives (Random Forest)',
                                      yaxis_title='Age')
            fig_err_age.show()
            logging.info("Displayed age distribution for FP vs FN.")
        else:
            logging.warning("Not enough false positives or false negatives to plot age distribution.")

else:
    logging.error("Test data or trained models not available for misclassification analysis.")

2025-10-26 16:17:13,986 - INFO - Visualizing Confusion Matrices for the models.


2025-10-26 16:17:15,526 - INFO - Displayed confusion matrices for all models.
2025-10-26 16:17:15,590 - INFO - 
--- Analysis of Misclassified Samples (Random Forest) ---
2025-10-26 16:17:15,593 - INFO - Total misclassified samples: 206
2025-10-26 16:17:15,593 - INFO - Number of False Positives (Actual 0, Predicted 1): 67
2025-10-26 16:17:15,593 - INFO - Number of False Negatives (Actual 1, Predicted 0): 139


Misclassified Samples (Actual vs Predicted):
       Actual  Predicted
3817       1          0
391        1          0
3874       1          0
1458       1          0
3510       0          1


2025-10-26 16:17:15,764 - INFO - Displayed age distribution for FP vs FN.


In [51]:
if 'scaler' in locals() and 'X' in locals() and trained_models:
    # Identify the best model based on F1-score from previous evaluation
    best_model_name = max(results, key=lambda k: results[k]['F1-Score'])
    best_model = trained_models[best_model_name]
    logging.info(f"Selected '{best_model_name}' as the best model for demonstration.")

    # Create a sample dataset with realistic values
    # Ensure all original feature names are present and in the correct order
    # from the original X (before scaling)
    sample_data = pd.DataFrame([
        {'male': 1, 'age': 55, 'education': 2.0, 'currentSmoker': 1, 'cigsPerDay': 20.0, 'BPMeds': 0.0,
         'prevalentStroke': 0, 'prevalentHyp': 1, 'diabetes': 0, 'totChol': 240.0, 'sysBP': 145.0,
         'diaBP': 90.0, 'BMI': 28.0, 'heartRate': 75.0, 'glucose': 95.0},
        {'male': 0, 'age': 40, 'education': 4.0, 'currentSmoker': 0, 'cigsPerDay': 0.0, 'BPMeds': 0.0,
         'prevalentStroke': 0, 'prevalentHyp': 0, 'diabetes': 0, 'totChol': 180.0, 'sysBP': 110.0,
         'diaBP': 70.0, 'BMI': 22.0, 'heartRate': 68.0, 'glucose': 80.0},
        {'male': 1, 'age': 68, 'education': 1.0, 'currentSmoker': 1, 'cigsPerDay': 30.0, 'BPMeds': 1.0,
         'prevalentStroke': 0, 'prevalentHyp': 1, 'diabetes': 1, 'totChol': 280.0, 'sysBP': 160.0,
         'diaBP': 100.0, 'BMI': 32.0, 'heartRate': 90.0, 'glucose': 120.0}
    ], columns=X.columns) # Use original columns to maintain order

    logging.info("Created a sample dataset for prediction:")
    print(sample_data)

    try:
        # Scale the numerical features of the sample data using the *fitted* scaler
        sample_data_scaled = sample_data.copy()
        sample_data_scaled[numerical_features] = scaler.transform(sample_data[numerical_features])
        logging.info("Sample data scaled.")

        # Make predictions
        sample_predictions = best_model.predict(sample_data_scaled)
        sample_probabilities = best_model.predict_proba(sample_data_scaled)[:, 1] if hasattr(best_model, 'predict_proba') else ['N/A'] * len(sample_data_scaled)

        logging.info("\nPredictions on the sample dataset:")
        for i, (pred, prob) in enumerate(zip(sample_predictions, sample_probabilities)):
            prediction_text = "Positive (CHD risk)" if pred == 1 else "Negative (No CHD risk)"
            prob_text = f" (Probability: {prob:.4f})" if isinstance(prob, float) else ""
            logging.info(f"Sample {i+1}: Predicted class: {prediction_text}{prob_text}")
    except Exception as e:
        logging.error(f"Error making predictions on sample data: {e}")
else:
    logging.error("Scaler or best model not available for new data prediction.")

2025-10-26 16:17:46,927 - INFO - Selected 'Logistic Regression' as the best model for demonstration.
2025-10-26 16:17:46,936 - INFO - Created a sample dataset for prediction:
2025-10-26 16:17:46,975 - INFO - Sample data scaled.
2025-10-26 16:17:46,996 - INFO - 
Predictions on the sample dataset:
2025-10-26 16:17:46,999 - INFO - Sample 1: Predicted class: Positive (CHD risk) (Probability: 0.6964)
2025-10-26 16:17:46,999 - INFO - Sample 2: Predicted class: Negative (No CHD risk) (Probability: 0.1188)
2025-10-26 16:17:47,008 - INFO - Sample 3: Predicted class: Positive (CHD risk) (Probability: 0.9799)


   male  age  education  currentSmoker  cigsPerDay  BPMeds  prevalentStroke  \
0     1   55        2.0              1        20.0     0.0                0   
1     0   40        4.0              0         0.0     0.0                0   
2     1   68        1.0              1        30.0     1.0                0   

   prevalentHyp  diabetes  totChol  sysBP  diaBP   BMI  heartRate  glucose  
0             1         0    240.0  145.0   90.0  28.0       75.0     95.0  
1             0         0    180.0  110.0   70.0  22.0       68.0     80.0  
2             1         1    280.0  160.0  100.0  32.0       90.0    120.0  


In [52]:
if 'X_train_final' in locals() and 'y_train_final' in locals():
    logging.info("\n--- Starting Hyperparameter Tuning for Random Forest Classifier ---")

    # Use a smaller subset of the training data for faster tuning as per instruction
    # For a full run, comment out this subsampling
    X_train_tuning, _, y_train_tuning, _ = train_test_split(X_train_final, y_train_final, test_size=0.8, random_state=42, stratify=y_train_final)
    logging.info(f"Using a subset of training data for tuning: {X_train_tuning.shape}")

    # Define the parameter grid
    param_grid = {
        'n_estimators': [50, 100], # Reduced for speed
        'max_features': ['sqrt', None], # 'auto' is now 'sqrt'
        'max_depth': [5, 10], # Reduced for speed
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    }

    rf_model_tuning = RandomForestClassifier(random_state=42)

    try:
        # Initialize GridSearchCV
        grid_search = GridSearchCV(estimator=rf_model_tuning, param_grid=param_grid,
                                   cv=3, n_jobs=-1, verbose=1, scoring='f1', error_score='raise') # cv=3 for speed

        # Fit GridSearchCV
        grid_search.fit(X_train_tuning, y_train_tuning)

        logging.info("Hyperparameter tuning complete.")
        logging.info(f"Best parameters found: {grid_search.best_params_}")
        logging.info(f"Best F1-score (cross-validated): {grid_search.best_score_:.4f}")

        # Update the best model with tuned parameters
        tuned_rf_model = grid_search.best_estimator_
        trained_models['Tuned Random Forest'] = tuned_rf_model
        logging.info("Tuned Random Forest Classifier added to trained models.")

        # Re-evaluate all models including the tuned one to see improvement
        logging.info("\nRe-evaluating models including the tuned Random Forest...")
        results = {}
        roc_curves = {}
        for name, model in trained_models.items():
            try:
                y_pred = model.predict(X_test_scaled)
                y_proba = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else [0] * len(y_test)

                accuracy = accuracy_score(y_test, y_pred)
                precision = precision_score(y_test, y_pred, zero_division=0)
                recall = recall_score(y_test, y_pred, zero_division=0)
                f1 = f1_score(y_test, y_pred, zero_division=0)
                roc_auc = roc_auc_score(y_test, y_proba) if hasattr(model, 'predict_proba') else 0

                results[name] = {
                    'Accuracy': accuracy,
                    'Precision': precision,
                    'Recall': recall,
                    'F1-Score': f1,
                    'ROC AUC': roc_auc
                }
                if hasattr(model, 'predict_proba'):
                    fpr, tpr, _ = roc_curve(y_test, y_proba)
                    roc_curves[name] = {'fpr': fpr, 'tpr': tpr, 'auc': roc_auc}

            except Exception as e:
                logging.error(f"Error during re-evaluation of {name} after tuning: {e}")

        results_df_tuned = pd.DataFrame(results).T
        print("\n--- Model Evaluation Results (After Tuning) ---")
        print(results_df_tuned.sort_values(by='F1-Score', ascending=False))

    except Exception as e:
        logging.error(f"Error during hyperparameter tuning: {e}")
else:
    logging.error("Training data not available for hyperparameter tuning.")

2025-10-26 16:18:13,977 - INFO - 
--- Starting Hyperparameter Tuning for Random Forest Classifier ---
2025-10-26 16:18:14,003 - INFO - Using a subset of training data for tuning: (1078, 15)


Fitting 3 folds for each of 32 candidates, totalling 96 fits


2025-10-26 16:18:23,757 - INFO - Hyperparameter tuning complete.
2025-10-26 16:18:23,757 - INFO - Best parameters found: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}
2025-10-26 16:18:23,757 - INFO - Best F1-score (cross-validated): 0.7725
2025-10-26 16:18:23,757 - INFO - Tuned Random Forest Classifier added to trained models.
2025-10-26 16:18:23,757 - INFO - 
Re-evaluating models including the tuned Random Forest...



--- Model Evaluation Results (After Tuning) ---
                              Accuracy  Precision    Recall  F1-Score  \
Logistic Regression           0.655660   0.237113  0.571429  0.335155   
Tuned Random Forest           0.715094   0.245487  0.422360  0.310502   
Support Vector Classifier     0.691509   0.233974  0.453416  0.308668   
Gradient Boosting Classifier  0.773585   0.271676  0.291925  0.281437   
Random Forest Classifier      0.805660   0.247191  0.136646  0.176000   

                               ROC AUC  
Logistic Regression           0.698091  
Tuned Random Forest           0.663346  
Support Vector Classifier     0.631772  
Gradient Boosting Classifier  0.645002  
Random Forest Classifier      0.624303  


In [53]:
if 'results_df_tuned' in locals():
    # Find the model with the highest F1-Score
    final_best_model_name = results_df_tuned['F1-Score'].idxmax()
    final_best_model = trained_models[final_best_model_name]

    logging.info(f"\n--- Final Model Selection ---")
    logging.info(f"The best performing model based on F1-Score is: '{final_best_model_name}'")
    logging.info(f"Its performance metrics are:\n{results_df_tuned.loc[final_best_model_name]}")

    selected_model_for_saving = final_best_model
else:
    logging.error("Model evaluation results not available for final model selection.")
    selected_model_for_saving = None # Fallback if results are missing


2025-10-26 16:19:31,157 - INFO - 
--- Final Model Selection ---
2025-10-26 16:19:31,162 - INFO - The best performing model based on F1-Score is: 'Logistic Regression'
2025-10-26 16:19:31,167 - INFO - Its performance metrics are:
Accuracy     0.655660
Precision    0.237113
Recall       0.571429
F1-Score     0.335155
ROC AUC      0.698091
Name: Logistic Regression, dtype: float64


In [55]:
ARTIFACTS_DIR = 'artifacts'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
logging.info(f"Artifacts directory created: {ARTIFACTS_DIR}")


if selected_model_for_saving is not None:
    # Define the path to save the model
    model_filename = f'{final_best_model_name.replace(" ", "_").lower()}_chd_model.pkl'
    model_filepath = os.path.join(ARTIFACTS_DIR, model_filename)

    try:
        with open(model_filepath, 'wb') as file:
            pickle.dump(selected_model_for_saving, file)
        logging.info(f"Final model '{final_best_model_name}' successfully saved to '{model_filepath}'")
    except Exception as e:
        logging.error(f"Error saving the model: {e}")
else:
    logging.error("No model selected for saving. Skipping model saving step.")

2025-10-26 16:20:24,071 - INFO - Artifacts directory created: artifacts
2025-10-26 16:20:24,073 - INFO - Final model 'Logistic Regression' successfully saved to 'artifacts\logistic_regression_chd_model.pkl'
